In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("ratings_baseline_label_encoded.csv")



In [4]:
X = df[['user_enc', 'movie_enc']].values
y = df['rating'].values
ti_le_train = int(0.8 * len(X))

X_train = X[:ti_le_train]
X_test  = X[ti_le_train:]

y_train = y[:ti_le_train]
y_test  = y[ti_le_train:]

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)


Train size: (80000, 2)
Test size : (20000, 2)


In [5]:
def khoang_cach_euclid(a, b):
    return np.sqrt(np.sum((a - b) ** 2))


In [ ]:
def du_doan_knn(x, X_train, y_train, k=20):
    danh_sach = []

    for i in range(len(X_train)):
        d = khoang_cach_euclid(x, X_train[i])
        danh_sach.append((d, y_train[i]))

    danh_sach.sort(key=lambda x: x[0])
    k_lang_gieng = danh_sach[:k]

    return np.mean([rating for _, rating in k_lang_gieng])


y_pred_knn = []
for x in X_test:
    y_pred_knn.append(du_doan_knn(x, X_train, y_train, k=20))

y_pred_knn = np.array(y_pred_knn)


In [ ]:
rmse_knn = np.sqrt(np.mean((y_test - y_pred_knn) ** 2))
mae_knn  = np.mean(np.abs(y_test - y_pred_knn))

print("=== KNN REGRESSOR (GIẢI TAY) ===")
print("RMSE:", rmse_knn)
print("MAE :", mae_knn)


In [ ]:
def train_cay_don_gian(X, y, chi_so_feature):
    phuong_sai = np.var(X[:, chi_so_feature], axis=0)
    feature_tot_nhat = chi_so_feature[np.argmax(phuong_sai)]

    nguong = np.median(X[:, feature_tot_nhat])

    ben_trai = y[X[:, feature_tot_nhat] <= nguong]
    ben_phai = y[X[:, feature_tot_nhat] > nguong]

    gia_tri_trai = np.mean(ben_trai) if len(ben_trai) > 0 else np.mean(y)
    gia_tri_phai = np.mean(ben_phai) if len(ben_phai) > 0 else np.mean(y)

    return feature_tot_nhat, nguong, gia_tri_trai, gia_tri_phai


def du_doan_cay_don_gian(X, cay):
    feature, nguong, trai, phai = cay
    return np.array([trai if x[feature] <= nguong else phai for x in X])


In [ ]:
def train_rung_ngau_nhien(X, y, so_cay=20):
    rung = []
    so_mau, so_feature = X.shape
    so_feature_ngau_nhien = int(np.sqrt(so_feature))

    for _ in range(so_cay):
        chi_so_mau = np.random.choice(so_mau, so_mau, replace=True)

        chi_so_feature = np.random.choice(
            so_feature,
            so_feature_ngau_nhien,
            replace=False
        )

        cay = train_cay_don_gian(
            X[chi_so_mau],
            y[chi_so_mau],
            chi_so_feature
        )

        rung.append(cay)

    return rung


def du_doan_rung_ngau_nhien(X, rung):
    tat_ca_du_doan = np.array([
        du_doan_cay_don_gian(X, cay) for cay in rung
    ])
    return np.mean(tat_ca_du_doan, axis=0)


# Train & predict
rung = train_rung_ngau_nhien(X_train, y_train, so_cay=20)
y_pred_rf = du_doan_rung_ngau_nhien(X_test, rung)


In [ ]:
rmse_rf = np.sqrt(np.mean((y_test - y_pred_rf) ** 2))
mae_rf  = np.mean(np.abs(y_test - y_pred_rf))

print("=== RANDOM FOREST REGRESSOR (GIẢI TAY) ===")
print("RMSE:", rmse_rf)
print("MAE :", mae_rf)


In [ ]:
ten_mo_hinh = [
    "KNN Regressor\n(Base Model)",
    "Random Forest Regressor\n(Ensemble Model)"
]

gia_tri_rmse = [
    rmse_knn,
    rmse_rf
]

plt.figure(figsize=(7, 5))

cot = plt.bar(
    ten_mo_hinh,
    gia_tri_rmse,
    width=0.5
)

plt.ylabel("RMSE", fontsize=11)
plt.xlabel("Mô hình", fontsize=11)
plt.title(
    "So sánh RMSE giữa mô hình đơn và mô hình tổ hợp (Giải tay)",
    fontsize=13
)

for c in cot:
    chieu_cao = c.get_height()
    plt.text(
        c.get_x() + c.get_width() / 2,
        chieu_cao,
        f"{chieu_cao:.3f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
